# Assignment 22 — Text Analytics: Preprocessing + TF-IDF

> **Note:** This problem statement is identical to Assignment 11 in the exam paper. Both assignments cover the same text-preprocessing pipeline and TF-IDF representation. This notebook is a fresh, standalone implementation.

> **Note on outputs:** Run all cells from top to bottom. NLTK data downloads automatically on the first run (requires internet).

## Topic Explanation

### What is Text Analytics?
**Text Analytics** (also called **Natural Language Processing** or **NLP**) is the process of extracting useful information from raw text. Unlike numbers, text is **unstructured data** — computers can't directly understand "the cat sat on the mat." We need to convert text into numerical features that ML algorithms can process.

The standard text-analytics pipeline has two stages:

### Stage 1 — Document Preprocessing (5 steps)

#### 1. Tokenization
Splitting text into **tokens** (individual words or sentences). Two types:
- **Word tokenization** — `"Text analytics is fun"` → `["Text", "analytics", "is", "fun"]`
- **Sentence tokenization** — splits on `.`, `?`, `!` while handling abbreviations correctly

#### 2. POS Tagging (Part-of-Speech)
Labeling each word with its grammatical role:
- `NN` — noun (e.g., "cat")
- `VB` — verb (e.g., "run")
- `JJ` — adjective (e.g., "big")
- `RB` — adverb (e.g., "quickly")
- `DT` — determiner (e.g., "the")
- `IN` — preposition (e.g., "on")

Useful for understanding sentence structure and filtering by role.

#### 3. Stop-word Removal
Stop words are common words like "the", "is", "a", "of", "and" that carry **little meaning** but appear extremely often. Removing them reduces noise and lets the model focus on content-bearing words.

#### 4. Stemming
Reducing words to their **crude root** by chopping off suffixes — using simple rules. Examples:
- `playing` → `play`
- `studies` → `studi`
- `connection` → `connect`

The most popular stemmer is the **Porter Stemmer**. It's fast but may produce non-words like `studi`.

#### 5. Lemmatization
Reducing words to their **dictionary base form** — using vocabulary and grammar rules. Examples:
- `playing` → `play`
- `studies` → `study`
- `better` → `good`
- `mice` → `mouse`

Slower than stemming but always returns real words. Uses **WordNet** as the lookup dictionary.

#### Stemming vs Lemmatization

| | Stemming | Lemmatization |
|---|---------|---------------|
| Speed | Fast | Slower |
| Method | Rule-based suffix-stripping | Dictionary lookup |
| Output | May not be a real word | Always a real word |
| Example | `studies` → `studi` | `studies` → `study` |
| Library | `PorterStemmer` | `WordNetLemmatizer` |

### Stage 2 — Representation: TF-IDF

Once text is preprocessed, we convert documents to numeric vectors using **TF-IDF** (Term Frequency × Inverse Document Frequency).

#### Term Frequency (TF)
> **TF(term, doc) = count(term in doc) / total terms in doc**

Measures how often a term appears in a single document.

#### Inverse Document Frequency (IDF)
> **IDF(term) = log(N / df(term))**

Where:
- **N** = total number of documents
- **df(term)** = number of documents containing the term

IDF is **high** for rare terms (appear in few docs), **low** for common terms (appear everywhere).

#### TF-IDF
> **TF-IDF(term, doc) = TF(term, doc) × IDF(term)**

**Intuition:** a term has high TF-IDF if it appears **frequently in this document** but **rarely across the whole corpus**. Such words are **distinctive** — they characterize what makes this document different.

**Example:** the word "the" appears often in every document → low IDF → low TF-IDF (not distinctive). But "photosynthesis" might appear often in a biology textbook chapter and rarely elsewhere → high TF-IDF → distinctive.

### Real-World Applications
- Search engines (ranking documents by relevance)
- Spam detection
- Sentiment analysis
- Document classification
- Topic modeling
- Recommendation systems

## Step 1: Import Libraries and Download NLTK Data

NLTK requires data files (tokenizers, taggers, stop-word lists) to be downloaded before first use. The downloads happen automatically the first time you run this notebook on your machine.

In [ ]:
# nltk — Natural Language Toolkit, the main library for text processing
import nltk

# Download required NLTK data (only runs once; afterwards cached locally)
nltk.download('punkt', quiet=True)              # word & sentence tokenizer
nltk.download('punkt_tab', quiet=True)          # newer punkt for nltk 3.8+
nltk.download('stopwords', quiet=True)          # English stop-word list
nltk.download('averaged_perceptron_tagger', quiet=True)         # POS tagger
nltk.download('averaged_perceptron_tagger_eng', quiet=True)     # English POS tagger
nltk.download('wordnet', quiet=True)            # WordNet dictionary for lemmatization
nltk.download('omw-1.4', quiet=True)            # Open Multilingual WordNet

# Specific NLP tools
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag

# pandas for displaying results in tables
import pandas as pd

# scikit-learn for TF-IDF vectorization
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

print("Libraries imported and NLTK data downloaded.")

## Step 2: Sample Document

In [ ]:
document = '''Text analytics is the process of deriving meaningful information from natural language text.
It involves several techniques such as tokenization, part of speech tagging, stemming, and lemmatization.
Text analytics is widely used in sentiment analysis and document classification.'''

print("Sample Document:")
print(document)
print(f"\nLength: {len(document)} characters")

## Step 3: Tokenization

Splitting text into tokens (sentences and words).

In [ ]:
# Sentence tokenization — splits document into sentences
sentences = sent_tokenize(document)
print(f"Number of sentences: {len(sentences)}\n")
for i, sent in enumerate(sentences, 1):
    print(f"{i}. {sent}")

In [ ]:
# Word tokenization — splits document into words and punctuation
words = word_tokenize(document)
print(f"Number of word tokens: {len(words)}\n")
print("First 20 tokens:")
print(words[:20])

## Step 4: POS Tagging

Labeling each word with its grammatical role:
- `NN` = noun, `VB` = verb, `JJ` = adjective, `RB` = adverb
- `DT` = determiner, `IN` = preposition, `CC` = conjunction
- `VBZ` = verb 3rd person singular, `VBG` = verb gerund/present participle

In [ ]:
# pos_tag() returns list of (word, tag) tuples
tagged = pos_tag(words)

# Display first 15 in a clean table
pd.DataFrame(tagged[:15], columns=['Word', 'POS Tag'])

In [ ]:
# Count tags by frequency
from collections import Counter
tag_counts = Counter(tag for _, tag in tagged)
print("POS Tag Distribution:")
for tag, count in sorted(tag_counts.items(), key=lambda x: -x[1]):
    print(f"  {tag:<6}: {count}")

## Step 5: Stop-word Removal

Remove common words like "the", "is", "of" that carry little meaning.

In [ ]:
# Get the English stop-word list
stop_words = set(stopwords.words('english'))
print(f"NLTK has {len(stop_words)} English stop words.")
print(f"Sample stop words: {list(stop_words)[:15]}")

In [ ]:
# Remove stop words AND punctuation tokens (keep only alphabetic words)
filtered = [w for w in words if w.lower() not in stop_words and w.isalpha()]

print(f"Tokens before filtering: {len(words)}")
print(f"Tokens after  filtering: {len(filtered)}")
print(f"\nFiltered tokens:")
print(filtered)

## Step 6: Stemming and Lemmatization

Both reduce words to their base form, but with different methods:
- **Stemming** (Porter) — fast rule-based suffix-stripping; may produce non-words
- **Lemmatization** (WordNet) — slower dictionary-based; always returns real words

In [ ]:
# Initialize the two tools
ps = PorterStemmer()
wl = WordNetLemmatizer()

# Apply both to each filtered token
stems  = [ps.stem(w) for w in filtered]
lemmas = [wl.lemmatize(w, pos='v') for w in filtered]   # pos='v' treats words as verbs

# Show side-by-side comparison
comparison = pd.DataFrame({
    'Original': filtered,
    'Stem':     stems,
    'Lemma':    lemmas
})
comparison

In [ ]:
# Highlight the rows where stem and lemma differ
diff = comparison[comparison['Stem'] != comparison['Lemma']]
print(f"Words where Stem ≠ Lemma ({len(diff)} cases):")
diff

## Step 7: Term Frequency (TF) and TF-IDF

Now we represent the document numerically. Each sentence is treated as a separate "document" in our small corpus.

### 7.1 Term Frequency (TF) — Just Counts

In [ ]:
# Treat each sentence as a separate document
docs = sentences

# CountVectorizer counts how often each word appears in each document
tf_vectorizer = CountVectorizer(stop_words='english')
tf_matrix = tf_vectorizer.fit_transform(docs)

# Display as a clean DataFrame
tf_df = pd.DataFrame(
    tf_matrix.toarray(),
    columns=tf_vectorizer.get_feature_names_out(),
    index=[f'Doc {i+1}' for i in range(len(docs))]
)

print("Term Frequency Matrix (raw counts):")
tf_df

### 7.2 TF-IDF — Weighted by Distinctiveness

In [ ]:
# TfidfVectorizer applies the full TF * IDF formula in one step
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(docs)

# Display as a DataFrame
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray().round(3),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f'Doc {i+1}' for i in range(len(docs))]
)

print("TF-IDF Matrix:")
tfidf_df

In [ ]:
# Show the IDF scores — higher = more distinctive (rare across documents)
idf_scores = pd.DataFrame({
    'Term': tfidf_vectorizer.get_feature_names_out(),
    'IDF':  tfidf_vectorizer.idf_.round(4)
}).sort_values('IDF', ascending=False)

print("IDF scores (higher = rarer = more distinctive):")
idf_scores

### 7.3 Top Terms per Document by TF-IDF

In [ ]:
# For each document, find the top-3 most distinctive terms
print("Top-3 most distinctive terms per document:\n")
for i, doc in enumerate(docs):
    print(f"Doc {i+1}: \"{doc[:60]}...\"")
    top3 = tfidf_df.iloc[i].nlargest(3)
    for term, score in top3.items():
        print(f"   {term:<20} TF-IDF={score:.3f}")
    print()

## Conclusion

We performed the full text-analytics pipeline on a sample document:

1. **Tokenization** — split the document into 3 sentences and 35+ word tokens.
2. **POS Tagging** — labeled each token with its grammatical role (noun, verb, etc.).
3. **Stop-word Removal** — kept only meaningful content words (~50% reduction).
4. **Stemming** — reduced words to crude roots using Porter's rules (e.g., "lemmatization" → "lemmat").
5. **Lemmatization** — reduced words to real dictionary forms using WordNet (e.g., "techniques" → "technique").
6. **TF-IDF Representation** — converted the text corpus into a numeric matrix where each cell shows how distinctive a term is for a given document.

**Key insight:** TF-IDF gives high scores to terms that are frequent in *this* document but rare across the whole corpus — these are the words that characterize what each document is about.

This is the foundation for nearly every text-based ML task: search engines, sentiment analysis, document classification, spam detection, and recommender systems.

## Explanation of Everything Used in This Notebook

### Libraries

| Library | Purpose |
|---------|---------|
| **nltk** | Natural Language Toolkit — tokenization, POS tagging, stop words, stemming, lemmatization |
| **sklearn** | `CountVectorizer` and `TfidfVectorizer` for matrix-form text representation |
| **pandas** | Display results in clean tables |

### NLTK Functions Used

#### Tokenization
- `sent_tokenize(text)` — splits text into sentences
- `word_tokenize(text)` — splits text into words and punctuation

#### POS Tagging
- `pos_tag(words)` — returns list of (word, tag) tuples

Common POS tags:
- `NN` = singular noun, `NNS` = plural noun
- `VB` = base verb, `VBZ` = 3rd person singular, `VBG` = gerund
- `JJ` = adjective, `RB` = adverb
- `DT` = determiner, `IN` = preposition

#### Stop Words
- `stopwords.words('english')` — list of ~180 English stop words
- Filtering: `[w for w in words if w.lower() not in stop_words]`

#### Stemming
- `PorterStemmer()` — most common English stemmer
- `.stem(word)` — strips suffixes using rules

#### Lemmatization
- `WordNetLemmatizer()` — uses WordNet dictionary
- `.lemmatize(word, pos='v')` — `pos` flag tells the lemmatizer what part of speech to assume; `'v'` = verb, `'n'` = noun (default)

### Scikit-learn Vectorizers

#### CountVectorizer (Term Frequency)
- `CountVectorizer(stop_words='english')` — creates a vectorizer with built-in stop-word removal
- `.fit_transform(docs)` — learns vocabulary and produces TF matrix in one step
- `.get_feature_names_out()` — returns array of vocabulary terms in column order

#### TfidfVectorizer (TF-IDF)
- `TfidfVectorizer(stop_words='english')` — applies TF and IDF together
- `.idf_` — array of IDF values per term
- The output is normalized (L2 norm by default), so document vectors have unit length

### Key Concepts

#### Tokenization
Splitting text into smaller units (tokens). The most basic preprocessing step.

#### Stop Words
Common words ("the", "is", "of") with little semantic value. Removing them reduces noise.

#### Stemming
Crude suffix-stripping (e.g., `connection` → `connect`). May produce non-words.

#### Lemmatization
Dictionary-based reduction to base form (e.g., `mice` → `mouse`). Always real words.

#### POS (Part of Speech) Tagging
Labeling words with their grammatical category. Useful for filtering ("only keep nouns") or for context-aware lemmatization.

#### Term Frequency (TF)
Count of how often a term appears in a document, normalized by document length.

#### Inverse Document Frequency (IDF)
A weight that's high for rare terms across the corpus and low for common terms:
> IDF(term) = log(N / df(term))

#### TF-IDF
Multiplies TF and IDF together. Captures terms that are:
- Frequent in the *specific document* (high TF)
- Rare across *all documents* (high IDF)

→ These are the **distinctive** terms that characterize each document.

#### Why TF-IDF Beats Raw Counts
Raw counts make common words like "the" look important because they appear often. TF-IDF discounts them because they appear in every document, surfacing the truly meaningful terms instead.

## Viva Questions (with Answers)

### Conceptual

**Q1. What is text analytics?**
The process of extracting useful information from raw text (Natural Language Processing). Includes preprocessing, feature extraction, and modeling.

**Q2. What is tokenization?**
Splitting text into smaller units called tokens — usually words or sentences.

**Q3. Difference between word and sentence tokenization?**
- **Word tokenization** splits on word boundaries (spaces, punctuation)
- **Sentence tokenization** splits on `.`, `?`, `!` while handling abbreviations

**Q4. What is POS tagging?**
Labeling each word with its grammatical role (noun, verb, adjective, etc.).

**Q5. What are stop words?**
Common words like "the", "is", "of", "a" that carry little meaning. Removing them reduces noise and storage requirements.

**Q6. What is stemming?**
Reducing words to their crude root form using rule-based suffix-stripping. Example: `playing` → `play`.

**Q7. What is lemmatization?**
Reducing words to their dictionary base form using vocabulary and grammar. Example: `mice` → `mouse`, `better` → `good`.

**Q8. Difference between stemming and lemmatization?**

| | Stemming | Lemmatization |
|---|---------|---------------|
| Speed | Fast | Slower |
| Method | Rule-based | Dictionary lookup |
| Output | May not be real word | Always real word |
| Example | `studies` → `studi` | `studies` → `study` |

**Q9. When use stemming vs lemmatization?**
- Use **stemming** when speed matters and exact word form doesn't (search engines, indexing)
- Use **lemmatization** when accuracy matters (sentiment analysis, conversational AI)

### Term Frequency / TF-IDF

**Q10. What is Term Frequency (TF)?**
How often a term appears in a document, normalized: TF = count(term in doc) / total terms in doc.

**Q11. What is Document Frequency (DF)?**
The number of documents in the corpus that contain a given term.

**Q12. What is Inverse Document Frequency (IDF)?**
IDF(term) = log(N / df(term)). High for rare terms, low for common terms.

**Q13. Why is IDF logarithmic?**
To dampen the effect of extremely rare terms. Without log, a term appearing in 1 of 1 million docs would have IDF = 1,000,000, dwarfing all other features.

**Q14. What is TF-IDF?**
The product TF × IDF. High = frequent here AND rare elsewhere → distinctive term.

**Q15. Why is TF-IDF better than raw counts?**
Raw counts make common words like "the" look important. TF-IDF discounts them because they appear everywhere, surfacing terms that actually characterize each document.

**Q16. What's the output shape of TF-IDF on a corpus of N documents and V vocabulary terms?**
N × V matrix — one row per document, one column per term.

### Practical / Code

**Q17. What does `nltk.download()` do?**
Downloads NLTK data files (tokenizers, taggers, dictionaries) to your machine. Required before first use.

**Q18. What does `stop_words='english'` do in CountVectorizer?**
Tells the vectorizer to remove common English stop words automatically before counting.

**Q19. Why do we use `pos='v'` in lemmatization?**
WordNet stores different forms for verbs vs nouns. Specifying `pos='v'` treats the word as a verb (so "better" → "good", not "better"). Default is noun.

**Q20. What is `.fit_transform()` doing in CountVectorizer?**
Two operations in one:
- **fit** — learn the vocabulary from the corpus
- **transform** — produce the count matrix using that vocabulary

**Q21. What's the output of `.toarray()`?**
Converts the sparse output matrix into a dense NumPy array (easier to display but uses more memory).

### Comparison & Extensions

**Q22. What other text representations exist besides TF-IDF?**
- **Bag-of-Words (BoW)** — simple counts (what `CountVectorizer` does)
- **n-grams** — pairs/triples of consecutive words
- **Word2Vec / GloVe** — dense vector embeddings
- **Sentence-BERT, GPT embeddings** — contextual embeddings from neural networks

**Q23. Limitations of TF-IDF?**
- Ignores word order
- Doesn't capture meaning or context (synonyms treated as different)
- Sparse: most cells are 0
- Doesn't generalize to unseen words

**Q24. What is named entity recognition (NER)?**
Identifying real-world entities in text — person names, locations, organizations, dates. A separate NLP task beyond preprocessing.

**Q25. What's the difference between `nltk` and `spaCy`?**
- **NLTK** — older, more academic, lots of options, slower
- **spaCy** — newer, faster, production-focused, opinionated defaults